Haremos lo mismo que en el otro fichero push-up-classification-eda.ipynb per aplicando a los videos transformacion espejo (tendremos 200 videos)

In [1]:
import sys
import os

# Ruta al directorio que contiene utils.py
sys.path.append(os.path.abspath("../src"))

from utils import extract_xy_sequence, extract_frame_from_video, draw_landmarks_on_frame

In [2]:
import cv2
import os

# Función mirror_video
def mirror_video(input_path, output_path):
    """
    Voltea un video horizontalmente (efecto espejo) y lo guarda.

    Args:
        input_path (str): Ruta al archivo de video original.
        output_path (str): Ruta donde se guardará el nuevo video espejado.
    """
    # 1. Abrir el video original
    cap = cv2.VideoCapture(input_path)

    if not cap.isOpened():
        print(f"Error: No se pudo abrir el video en {input_path}")
        return

    # 2. Obtener las propiedades del video
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # 3. Crear el objeto VideoWriter para el nuevo video
    # Usamos el mismo codec (MJPG es común y compatible)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    print(f"Procesando video: {input_path}")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 4. APLICAR LA TRANSFORMACIÓN ESPEJO
        # El código '1' indica un volteo horizontal
        mirrored_frame = cv2.flip(frame, 1)

        # 5. Escribir el fotograma espejado en el nuevo archivo
        out.write(mirrored_frame)

    # 6. Liberar recursos
    cap.release()
    out.release()
    print(f"✅ Video espejado guardado en: {output_path}")

# ======================================================================
# FUNCIÓN DE AUTOMATIZACIÓN
# ======================================================================

def process_all_videos(base_input_dir, base_output_dir):
    """
    Recorre las carpetas de videos originales, aplica el espejo
    y guarda los resultados manteniendo la estructura.
    """
    # 1. Crear la carpeta de salida si no existe
    os.makedirs(base_output_dir, exist_ok=True)
    print(f"\n--- Procesando directorio: {base_input_dir} ---")
    
    # 2. Iterar sobre todos los archivos del directorio de ENTRADA
    for filename in os.listdir(base_input_dir): # <-- Iteramos directamente en BASE_INPUT_DIR
        
        # Verificar que sea un archivo de video
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            
            # Construir rutas
            input_path = os.path.join(base_input_dir, filename)
            
            # Crear el nuevo nombre del archivo: ejemplo_MIRROR.mp4
            name, ext = os.path.splitext(filename)
            mirrored_filename = f"{name}_MIRROR{ext}"
            output_path = os.path.join(base_output_dir, mirrored_filename)
            
            # 3. Llamar a la función de transformación
            mirror_video(input_path, output_path)

# ======================================================================
# EJECUCIÓN DEL SCRIPT
# ======================================================================

# Secuencia correcta
BASE_INPUT_DIR = '../Data/Correct Sequence'
BASE_OUTPUT_DIR = '../Data/Correct_Mirrored_Videos'

# Asegúrate de que el directorio de salida exista
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# Llama a la función principal
process_all_videos(BASE_INPUT_DIR, BASE_OUTPUT_DIR)

print("\n Proceso de aumento de datos espejo completado en la carpeta de videos correctos.")

# Secuencia incorrecta
BASE_INPUT_DIR = '../Data/Wrong Sequence' 
BASE_OUTPUT_DIR = '../Data/Wrong_Mirrored_Videos'
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
process_all_videos(BASE_INPUT_DIR, BASE_OUTPUT_DIR)
print("\n Proceso de aumento de datos espejo completado en la carpeta de videos incorrectos.")


--- Procesando directorio: ../Data/Correct Sequence ---
Procesando video: ../Data/Correct Sequence/Copy of push up 138.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push up 138_MIRROR.mp4
Procesando video: ../Data/Correct Sequence/Copy of push up 113.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push up 113_MIRROR.mp4
Procesando video: ../Data/Correct Sequence/Copy of push up 102.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push up 102_MIRROR.mp4
Procesando video: ../Data/Correct Sequence/Copy of push up 116.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push up 116_MIRROR.mp4
Procesando video: ../Data/Correct Sequence/Copy of push up 100.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push up 100_MIRROR.mp4
Procesando video: ../Data/Correct Sequence/Copy of push up 114.mp4
✅ Video espejado guardado en: ../Data/Correct_Mirrored_Videos/Copy of push u

In [3]:
import numpy as np
import os
import cv2
import mediapipe as mp # Necesitas esta importación para extract_xy_sequence

# Asumo que tu función 'extract_xy_sequence' ya está definida en tu entorno.
# Si no lo está, debes añadirla junto con las importaciones necesarias (mp_pose, etc.).

# --- 1. DEFINICIÓN DE PARES SIMÉTRICOS A INTERCAMBIAR ---
# Estos índices corresponden a pares Izquierda (L) y Derecha (R) de MediaPipe
# que se deben intercambiar para corregir el efecto espejo.
SWAP_PAIRS = [
    (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22), # Hombros, Brazos y Manos
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)            # Caderas, Piernas y Pies
]

# ======================================================================
# 2. FUNCIÓN DE CORRECCIÓN MATEMÁTICA (APLICAR SÓLO A DATOS ESPEJADOS)
# ======================================================================

def correct_mirrored_landmarks(sequence):
    """
    Ajusta la coordenada X (1-x) e intercambia los pares Izquierda/Derecha
    en una secuencia de landmarks que proviene de un video espejado.
    
    Args:
        sequence (np.array): Array de forma (num_frames, 66).
    """
    if sequence.size == 0:
        return sequence

    mirrored_seq = sequence.copy()
    
    # 1. Ajuste de la coordenada X: x_corregida = 1 - x_original
    # Aplica a todos los índices de la columna par (0, 2, 4, ... que son las coordenadas X)
    mirrored_seq[:, 0::2] = 1.0 - mirrored_seq[:, 0::2] 
    
    # 2. Intercambio de Pares Izquierda/Derecha (Swapping)
    for idx_L, idx_R in SWAP_PAIRS:
        # L_start es el índice de la coordenada X para el landmark Izquierdo (índice * 2)
        L_start, R_start = idx_L * 2, idx_R * 2
        L_end, R_end = L_start + 2, R_start + 2 # +2 para incluir X e Y
        
        # Intercambiar los valores (X_L, Y_L) con (X_R, Y_R)
        temp = mirrored_seq[:, L_start:L_end].copy()
        mirrored_seq[:, L_start:L_end] = mirrored_seq[:, R_start:R_end]
        mirrored_seq[:, R_start:R_end] = temp
        
    return mirrored_seq


In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences 

# ======================================================================
# 3. FUNCIÓN DE CARGA, EXTRACCIÓN Y CORRECCIÓN (AUTOMATIZACIÓN)
# ======================================================================

MAX_TIMESTEPS = 160

def process_single_mirrored_dir(directory_path, max_timesteps):
    """
    Procesa un directorio, extrae landmarks, corrige la simetría y aplica padding.
    """
    raw_sequences = [] 
    
    if not os.path.exists(directory_path):
        print(f"⚠️ Error: Directorio no encontrado: {directory_path}. Retornando vacío.")
        return np.array([]) 

    print(f"\n--- Procesando: {directory_path} ---")
    
    for filename in os.listdir(directory_path):
        if filename.lower().endswith(('.mp4', '.avi', '.mov')):
            video_path = os.path.join(directory_path, filename)
            
            # 1. PASO FALTANTE: Extraer la secuencia bruta
            seq_bruta = extract_xy_sequence(video_path) # <-- ¡Aquí se define seq_bruta!
            
            # 2. Corregir la secuencia (Ahora seq_bruta existe)
            seq_corrected = correct_mirrored_landmarks(seq_bruta) 
            raw_sequences.append(seq_corrected)

    if not raw_sequences:
        print("Advertencia: No se encontraron videos para procesar.")
        return np.array([])
        
    # 3. Aplicar Padding a las secuencias (Resuelve el ValueError anterior)
    padded_sequences = pad_sequences(
        raw_sequences, 
        maxlen=max_timesteps, 
        dtype='float32', 
        padding='post',
        value=0.0       
    )
    
    return padded_sequences


# ======================================================================
# 4. CARGA Y PROCESAMIENTO
# ======================================================================

MIRRORED_CORRECT_PATH = '../Data/Correct_Mirrored_Videos'
MIRRORED_WRONG_PATH = '../Data/Wrong_Mirrored_Videos'

# --- B. Procesamiento de Videos Espejados ---
# Llamamos a la función dos veces, una para cada categoría
X_correct_mirrored = process_single_mirrored_dir(MIRRORED_CORRECT_PATH, MAX_TIMESTEPS)
X_incorrect_mirrored = process_single_mirrored_dir(MIRRORED_WRONG_PATH, MAX_TIMESTEPS)


# ======================================================================
# 4. VERIFICACIÓN Y RESULTADOS
# ======================================================================

print(f"Total Espejados Correctos: {X_correct_mirrored.shape[0]}")
print(f"Total Espejados Incorrectos: {X_incorrect_mirrored.shape[0]}")


--- Procesando: ../Data/Correct_Mirrored_Videos ---


I0000 00:00:1765561044.154093  129709 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1765561044.226199  130983 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765561044.232651  130983 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/miniconda3/envs/exml-py310/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
I0000 00:00:1765561046.532400  129709 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1


--- Procesando: ../Data/Wrong_Mirrored_Videos ---


I0000 00:00:1765561125.037222  129709 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1765561125.107351  132562 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765561125.113559  132562 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765561127.054608  129709 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1765561127.125426  132586 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1765561127.132395  132593 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1765561128.711710  129709 gl

Total Espejados Correctos: 50
Total Espejados Incorrectos: 50


In [5]:
# Almacenamos en discos los arrays procesados
np.save("../Data/processed/mirrored_correct_push_ups_landmarks.npy", X_correct_mirrored)
np.save("../Data/processed/mirrored_wrong_push_ups_landmarks.npy",   X_incorrect_mirrored)